<a href="https://colab.research.google.com/github/Anyangos/Digital-Twin/blob/adding_agents/Week3_10_tool_use_ai_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [3]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [19]:
import collections
from typing import Dict, List, Any

# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

# Sample weekly activities dataset
WEEKLY_ACTIVITIES = [
    {"day": "Monday", "activity": "Morning jog", "category": "Exercise"},
    {"day": "Monday", "activity": "Read 10 pages of book", "category": "Reading"},
    {"day": "Monday", "activity": "Work on code project", "category": "Coding"},
    {"day": "Tuesday", "activity": "Gym session", "category": "Exercise"},
    {"day": "Tuesday", "activity": "Meditation", "category": "Mindfulness"},
    {"day": "Tuesday", "activity": "Read 10 pages of book", "category": "Reading"},
    {"day": "Wednesday", "activity": "Morning jog", "category": "Exercise"},
    {"day": "Wednesday", "activity": "Work on code project", "category": "Coding"},
    {"day": "Wednesday", "activity": "Read 10 pages of book", "category": "Reading"},
    {"day": "Thursday", "activity": "Gym session", "category": "Exercise"},
    {"day": "Thursday", "activity": "Meditation", "category": "Mindfulness"},
    {"day": "Thursday", "activity": "Read 10 pages of book", "category": "Reading"},
    {"day": "Friday", "activity": "Morning jog", "category": "Exercise"},
    {"day": "Friday", "activity": "Work on code project", "category": "Coding"},
    {"day": "Friday", "activity": "Read 10 pages of book", "category": "Reading"},
    {"day": "Saturday", "activity": "Hiking", "category": "Exercise"},
    {"day": "Saturday", "activity": "Read 10 pages of book", "category": "Reading"},
    {"day": "Sunday", "activity": "Rest day", "category": "Mindfulness"}
]

def habit_tracker(arg: str = "") -> Dict[str, Any]:
    '''Goes through weekly activities to determine and return the top 5 habits based on frequency.'''
    habit_counts = collections.Counter([act['activity'] for act in WEEKLY_ACTIVITIES])
    top_5_habits = habit_counts.most_common(5)
    result_list = [f"{habit}: {count} times a week" for habit, count in top_5_habits]
    return {
        'summary': "Here are your top habits extracted from your weekly activities.",
        'top_habits': result_list
    }

def daily_task_list(day_of_week: str) -> Dict[str, Any]:
    '''Creates a daily task list based on the day of the week requested.'''
    day_cleaned = day_of_week.strip().capitalize()
    tasks = [act['activity'] for act in WEEKLY_ACTIVITIES if act['day'].capitalize() == day_cleaned]
    if not tasks:
        return {'error': f"No activities found for '{day_of_week}'. Please provide a valid day of the week."}
    return {
        'day': day_cleaned,
        'summary': f"Your scheduled tasks for {day_cleaned}.",
        'tasks': tasks
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())
print('Task', daily_task_list('Monday'))
print('Habits:', habit_tracker())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '09:05:40', 'date': '2026-11-24', 'day': 'Tuesday'}
Task {'day': 'Monday', 'summary': 'Your scheduled tasks for Monday.', 'tasks': ['Morning jog', 'Read 10 pages of book', 'Work on code project']}
Habits: {'summary': 'Here are your top habits extracted from your weekly activities.', 'top_habits': ['Read 10 pages of book: 6 times a week', 'Morning jog: 3 times a week', 'Work on code project: 3 times a week', 'Gym session: 2 times a week', 'Meditation: 2 times a week']}


## Part 2: Tool Registry

Manage available tools.

In [12]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f" - {name}: {info['description']}")
        return '\n'.join(lines)

# Create registry and register all tools directly
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')
registry.register('habit_tracker', habit_tracker, 'Tracks and identifies top habits')
registry.register('daily_task_list', daily_task_list, 'Creates a daily task list from activities')

print(registry.list_tools())


Available tools:
 - calculator: Calculate math expressions
 - get_time: Get current date and time
 - habit_tracker: Tracks and identifies top habits
 - daily_task_list: Creates a daily task list from activities


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [26]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']) and not any(day.lower() in query_lower for day in ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday']):
            return 'use_time'
        elif any(word in query_lower for word in ['habit', 'routine', 'frequent']):
            return 'use_habit_tracker'
        elif any(word in query_lower for word in ['task', 'schedule', 'todo', 'to-do', 'agenda']):
            return 'use_task_list'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        elif action == 'use_habit_tracker':
            return self.registry.get_tool('habit_tracker')()

        elif action == 'use_task_list':
            days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
            target_day = "Monday"  # Default fallback
            for day in days:
                if day.lower() in query.lower():
                    target_day = day
                    break
            return self.registry.get_tool('daily_task_list')(target_day)

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        if 'result' in observation:
            return f"The result is {observation['result']}"
        elif 'time' in observation:
            return f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'top_habits' in observation:
            habits_str = '\n- '.join(observation['top_habits'])
            return f"{observation['summary']}\n- {habits_str}"
        elif 'tasks' in observation:
            tasks_str = '\n- '.join(observation['tasks'])
            return f"Here is your task list for {observation['day']}:\n- {tasks_str}"
        elif 'error' in observation:
            return f"Error: {observation['error']}"
        else:
            return "I can help with calculations, time, habits, and task lists."

# Test the base agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What time is it?')
print('A:', agent.run('What time is it?'))
print('Q: What should I do today?')
print('A:', agent.run('What is my agenda'))



Agent Tests:
Q: What time is it?
A: It's 09:11:07 on 2026-11-24 (Tuesday)
Q: What should I do today?
A: Here is your task list for Monday:
- Morning jog
- Read 10 pages of book
- Work on code project


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [7]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools and initialize the agents.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 08:08:12 on 2026-11-24 (Tuesday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [14]:
# Re-initialize the twin with our updated tools and agent directly
twin = DigitalTwinWithTools('My Assistant')

# Adding the tools directly
twin.add_tool('habit_tracker', habit_tracker, 'Tracks and identifies top habits')
twin.add_tool('daily_task_list', daily_task_list, 'Creates a daily task list from activities')

# Re-assign the freshly modified SimpleReActAgent which now natively supports the new tools
twin.agent = SimpleReActAgent(twin.registry)

# Test the integration through your Digital Twin
print("--- Integrated Twin Custom Tool Tests ---")
print("Test 1:", twin.chat("What are my top habits?"))
print("\nTest 2:", twin.chat("Create a task list for Sunday"))


--- Integrated Twin Custom Tool Tests ---
Test 1: Here are your top habits extracted from your weekly activities.
- Read 10 pages of book: 6 times a week
- Morning jog: 3 times a week
- Work on code project: 3 times a week
- Gym session: 2 times a week
- Meditation: 2 times a week

Test 2: Here is your task list for Sunday:
- Rest day


## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI